## SQL-based data processing.

This notebook demonstrates a SQL-based data preprocessing pipeline integrated with Python-based credit risk modeling.

**The workflow follows a structured pipeline:**

1. Raw data ingestion
2. Data cleaning and preprocessing using SQL
3. Feature engineering and construction of model-ready dataset
4. Data validation at multiple stages
5. Downstream risk modeling in Python

This structure reflects a typical workflow in credit risk analytics and model validation environments.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [8]:
# 1. read raw data
df_raw = pd.read_csv("/Users/quentingao/Desktop/Github_CreditRisk/german_credit_data.csv")

# 2. create sqlite connection
conn = sqlite3.connect(":memory:")

# 3. write raw data into sqlite
df_raw.to_sql("raw_credit", conn, index=False, if_exists="replace")

# quick check
pd.read_sql("SELECT * FROM raw_credit LIMIT 5;", conn)

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,0,67,male,2,own,None,little,1169,6,radio/TV
1,1,22,female,2,own,little,moderate,5951,48,radio/TV
2,2,49,male,1,own,little,None,2096,12,education
3,3,45,male,2,free,little,little,7882,42,furniture/equipment
4,4,53,male,2,free,little,little,4870,24,car


In [9]:
query_clean = """
CREATE TABLE cleaned_credit AS
SELECT
    Age,
    TRIM(Sex) AS Sex,
    Job,
    TRIM(Housing) AS Housing,
    CASE 
        WHEN TRIM("Saving accounts") IN ('nan', 'NA', '') THEN NULL
        ELSE TRIM("Saving accounts")
    END AS saving_accounts,
    CASE 
        WHEN TRIM("Checking account") IN ('nan', 'NA', '') THEN NULL
        ELSE TRIM("Checking account")
    END AS checking_account,
    "Credit amount" AS credit_amount,
    Duration,
    TRIM(Purpose) AS Purpose
FROM raw_credit;
"""
conn.execute(query_clean)

In [10]:
pd.read_sql("SELECT * FROM cleaned_credit LIMIT 10;", conn)

,Age,Sex,Job,Housing,saving_accounts,checking_account,credit_amount,Duration,Purpose
0,67,male,2,own,None,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,None,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car
5,35,male,1,free,None,None,9055,36,education
6,53,male,2,own,quite rich,None,2835,24,furniture/equipment
7,35,male,3,rent,little,moderate,6948,36,car
8,61,male,1,own,rich,None,3059,12,radio/TV
9,28,male,3,own,little,moderate,5234,30,car


In [17]:
pd.read_sql("""
SELECT
    SUM(CASE WHEN saving_accounts IS NULL THEN 1 ELSE 0 END) AS missing_saving,
    SUM(CASE WHEN checking_account IS NULL THEN 1 ELSE 0 END) AS missing_checking,
    SUM(CASE WHEN Housing IS NULL THEN 1 ELSE 0 END) AS missing_housing
FROM cleaned_credit;
""", conn)

,missing_saving,missing_checking,missing_housing
0,183,394,0


In [11]:
query_features = """
CREATE TABLE feature_ready_credit AS
SELECT
    *,
    
    CASE
        WHEN saving_accounts = 'little' THEN 0
        WHEN saving_accounts = 'moderate' THEN 1
        WHEN saving_accounts = 'quite rich' THEN 2
        WHEN saving_accounts = 'rich' THEN 3
        ELSE 0.5
    END AS saving_score,

    CASE
        WHEN checking_account = 'little' THEN 0
        WHEN checking_account = 'moderate' THEN 1
        WHEN checking_account = 'rich' THEN 2
        ELSE 0.5
    END AS checking_score,

    CASE
        WHEN Housing = 'rent' THEN 0
        WHEN Housing = 'free' THEN 1
        WHEN Housing = 'own' THEN 2
        ELSE NULL
    END AS housing_score,

    Job AS job_score
FROM cleaned_credit;
"""
conn.execute(query_features)

In [12]:
pd.read_sql("""
SELECT 
    Age, Housing, saving_accounts, checking_account,
    saving_score, checking_score, housing_score, job_score
FROM feature_ready_credit
LIMIT 10;
""", conn)

,Age,Housing,saving_accounts,checking_account,saving_score,checking_score,housing_score,job_score
0,67,own,None,little,0.5,0.0,2,2
1,22,own,little,moderate,0.0,1.0,2,2
2,49,own,little,None,0.0,0.5,2,1
3,45,free,little,little,0.0,0.0,1,2
4,53,free,little,little,0.0,0.0,1,2
5,35,free,None,None,0.5,0.5,1,1
6,53,own,quite rich,None,2.0,0.5,2,2
7,35,rent,little,moderate,0.0,1.0,0,3
8,61,own,rich,None,3.0,0.5,2,1
9,28,own,little,moderate,0.0,1.0,2,3


In [13]:
stats = pd.read_sql("""
SELECT
    AVG(Age) AS age_mean,
    AVG(credit_amount) AS credit_mean,
    AVG(Duration) AS duration_mean
FROM feature_ready_credit;
""", conn)

age_mean = stats.loc[0, "age_mean"]
credit_mean = stats.loc[0, "credit_mean"]
duration_mean = stats.loc[0, "duration_mean"]

# pandas算std，和你原来逻辑一致
tmp = pd.read_sql("SELECT Age, credit_amount, Duration FROM feature_ready_credit;", conn)
age_std = tmp["Age"].std()
credit_std = tmp["credit_amount"].std()
duration_std = tmp["Duration"].std()

In [14]:
query_model_ready = f"""
CREATE TABLE model_ready_credit AS
SELECT
    *,
    (Age - {age_mean}) / {age_std} AS Age_z,
    (credit_amount - {credit_mean}) / {credit_std} AS Credit_amount_z,
    (Duration - {duration_mean}) / {duration_std} AS Duration_z
FROM feature_ready_credit;
"""
conn.execute(query_model_ready)

In [15]:
pd.read_sql("""
SELECT
    Age, credit_amount, Duration,
    saving_score, checking_score, housing_score, job_score,
    Age_z, Credit_amount_z, Duration_z
FROM model_ready_credit
LIMIT 10;
""", conn)

,Age,credit_amount,Duration,saving_score,checking_score,housing_score,job_score,Age_z,Credit_amount_z,Duration_z
0,67,1169,6,0.5,0.0,2,2,2.765073,-0.744759,-1.235859
1,22,5951,48,0.0,1.0,2,2,-1.190808,0.949342,2.247070
2,49,2096,12,0.0,0.5,2,1,1.182721,-0.416354,-0.738298
3,45,7882,42,0.0,0.0,1,2,0.831087,1.633430,1.749509
4,53,4870,24,0.0,0.0,1,2,1.534354,0.566380,0.256825
5,35,9055,36,0.5,0.5,1,1,-0.047998,2.048984,1.251947
6,53,2835,24,2.0,0.5,2,2,1.534354,-0.154551,0.256825
7,35,6948,36,0.0,1.0,0,3,-0.047998,1.302545,1.251947
8,61,3059,12,3.0,0.5,2,1,2.237622,-0.075196,-0.738298
9,28,5234,30,0.0,1.0,2,3,-0.663357,0.695333,0.754386


In [18]:
pd.read_sql("""
SELECT
    COUNT(*) AS n_obs,
    AVG(credit_amount) AS avg_credit,
    MIN(credit_amount) AS min_credit,
    MAX(credit_amount) AS max_credit,
    AVG(Duration) AS avg_duration
FROM model_ready_credit;
""", conn)

,n_obs,avg_credit,min_credit,max_credit,avg_duration
0,1000,3271.258,250,18424,20.903


In [16]:
df_model = pd.read_sql("SELECT * FROM model_ready_credit;", conn)

df_model["risk_score"] = (
    0.60 * df_model["Credit_amount_z"]
    + 0.75 * df_model["Duration_z"]
    - 0.25 * df_model["Age_z"]
    - 0.80 * df_model["saving_score"]
    - 0.60 * df_model["checking_score"]
    - 0.30 * df_model["housing_score"]
    - 0.20 * df_model["job_score"]
)

df_model["pseudo_pd"] = 1 / (1 + np.exp(-df_model["risk_score"]))

df_model["risk_bucket"] = pd.cut(
    df_model["pseudo_pd"],
    bins=[0, 0.20, 0.40, 0.60, 1.00],
    labels=["Low", "Moderate", "Elevated", "High"],
    include_lowest=True
)

df_model.head()

,Age,Sex,Job,Housing,saving_accounts,checking_account,credit_amount,Duration,Purpose,saving_score,checking_score,housing_score,job_score,Age_z,Credit_amount_z,Duration_z,risk_score,pseudo_pd,risk_bucket
0,67,male,2,own,None,little,1169,6,radio/TV,0.5,0.0,2,2,2.765073,-0.744759,-1.235859,-3.465018,0.030324,Low
1,22,female,2,own,little,moderate,5951,48,radio/TV,0.0,1.0,2,2,-1.190808,0.949342,2.247070,0.952610,0.721640,High
2,49,male,1,own,little,None,2096,12,education,0.0,0.5,2,1,1.182721,-0.416354,-0.738298,-2.199216,0.099821,Low
3,45,male,2,free,little,little,7882,42,furniture/equipment,0.0,0.0,1,2,0.831087,1.633430,1.749509,1.384418,0.799700,High
4,53,male,2,free,little,little,4870,24,car,0.0,0.0,1,2,1.534354,0.566380,0.256825,-0.551142,0.365599,Moderate
